In [1]:
import os, json, subprocess, pathlib
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.fs as pafs
from google.cloud import bigquery, dataform_v1beta1 as dataform, storage

P8_BUCKET = "cs544-atoscano2-p8"
PROJECT = "cs544-jan26"
REGION = "us-central1"
DATASET = "p8"
REPO_ID = "p8-repo"
WORKSPACE_ID = "p8-workspace"

print(f"PROJECT={PROJECT}  BUCKET={P8_BUCKET}")

PROJECT=cs544-jan26  BUCKET=cs544-atoscano2-p8


In [2]:
#q1
pathlib.Path("/etc/os-release").read_text()

'PRETTY_NAME="Ubuntu 26.04 LTS"\nNAME="Ubuntu"\nVERSION_ID="26.04"\nVERSION="26.04 (Resolute Raccoon)"\nVERSION_CODENAME=resolute\nID=ubuntu\nID_LIKE=debian\nHOME_URL="https://www.ubuntu.com/"\nSUPPORT_URL="https://help.ubuntu.com/"\nBUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"\nPRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"\nUBUNTU_CODENAME=resolute\nLOGO=ubuntu-logo\n'

In [3]:
#q2
subprocess.check_output(["lscpu"], text=True)

'Architecture:                            x86_64\nCPU op-mode(s):                          32-bit, 64-bit\nAddress sizes:                           46 bits physical, 48 bits virtual\nByte Order:                              Little Endian\nCPU(s):                                  2\nOn-line CPU(s) list:                     0,1\nVendor ID:                               GenuineIntel\nModel name:                              Intel(R) Xeon(R) CPU @ 2.20GHz\nCPU family:                              6\nModel:                                   79\nThread(s) per core:                      2\nCore(s) per socket:                      1\nSocket(s):                               1\nStepping:                                0\nBogoMIPS:                                4400.41\nFlags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid

In [4]:
gcs = pafs.GcsFileSystem()

local_path = "wi-schools-raw.parquet"
dest_path = f"{P8_BUCKET}/wi-schools-raw.parquet"

with open(local_path, "rb") as f:
    raw = f.read()

with gcs.open_output_stream(dest_path) as out:
    out.write(raw)

print("Uploaded!")

Uploaded!


In [5]:
#q3
gcs_client = storage.Client()
blobs = list(gcs_client.list_blobs(P8_BUCKET))
paths = [f"gs://{P8_BUCKET}/{b.name}" for b in blobs]
paths

['gs://cs544-atoscano2-p8/wi-schools-raw.parquet']

In [6]:
#q4
target_blob = gcs_client.bucket(P8_BUCKET).blob("wi-schools-raw.parquet")
target_blob.reload()
mod_time_ns = int(target_blob.updated.timestamp() * 1e9)
mod_time_ns

1777258004477999872

In [7]:
bq = bigquery.Client(project=PROJECT)

dataset_ref = bigquery.Dataset(f"{PROJECT}.{DATASET}")
dataset_ref.location = "US"
try:
    bq.create_dataset(dataset_ref)
    print("Dataset created.")
except Exception as e:
    print(f"(already exists or error): {e}")

Dataset created.


In [14]:
REPO_PARENT = f"projects/{PROJECT}/locations/{REGION}/repositories/p8-repo"
WORKSPACE = f"{REPO_PARENT}/workspaces/p8-workspace"

dfm = dataform.DataformClient()

files = {
    "wi_counties.sqlx": open("definitions/wi_counties.sqlx", "rb").read(),
    "schools.sqlx": open("definitions/schools.sqlx", "rb").read(),
    "wi_county_schools.sqlx": open("definitions/wi_county_schools.sqlx", "rb").read(),
}

for fname, contents in files.items():
    dfm.write_file(dataform.WriteFileRequest(
        workspace=WORKSPACE,
        path=f"definitions/{fname}",
        contents=contents,
    ))
    print(f"Uploaded {fname}")

Uploaded wi_counties.sqlx
Uploaded schools.sqlx
Uploaded wi_county_schools.sqlx


In [15]:
compilation = dfm.create_compilation_result(
    dataform.CreateCompilationResultRequest(
        parent=REPO_PARENT,
        compilation_result=dataform.CompilationResult(
            workspace=WORKSPACE,
        ),
    )
)
assert compilation.compilation_errors == [], compilation.compilation_errors
compilation_result_name = compilation.name
print("Compiled OK:", compilation_result_name)

Compiled OK: projects/cs544-jan26/locations/us-central1/repositories/p8-repo/compilationResults/ca002c91-0133-48d9-bfdf-a8c9e9efb21f


In [16]:
#q5
response = dfm.query_compilation_result_actions(
    request={"name": compilation_result_name}
)

deps = {}
for action in response.compilation_result_actions:
    name = action.target.name
    dep_names = [d.name for d in action.relation.dependency_targets]
    deps[name] = dep_names

deps

{'first_view': [],
 'schools': [],
 'second_view': ['first_view'],
 'wi_counties': [],
 'wi_county_schools': ['schools', 'wi_counties']}

In [17]:
#q6
result = bq.query(f"""
    SELECT COUNT(*) AS n
    FROM `{PROJECT}.{DATASET}.wi_counties`
""").result()

int(list(result)[0]["n"])

72

In [23]:
#q7
result = bq.query(f"""
    SELECT COUNT(*) AS n
    FROM `{PROJECT}.{DATASET}.wi_county_schools`
    WHERE agency_type = 'Public school'
""").result()

int(list(result)[0]["n"])

2116

In [24]:
#q8
sql = f"""
FROM `{PROJECT}.{DATASET}.wi_county_schools`
|> WHERE agency_type = 'Public school' AND school_type = 'High School'
|> AGGREGATE COUNT(*) AS hs_count
   GROUP BY county_name, city
|> WHERE hs_count >= 3
|> AGGREGATE COUNT(*) AS city_count
   GROUP BY county_name
|> WHERE city_count >= 2
|> SELECT county_name
|> ORDER BY county_name
"""

rows = list(bq.query(sql).result())
[r["county_name"] for r in rows]

['Brown', 'Dane', 'Milwaukee', 'Waukesha']

In [25]:
#q9
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
sql_q8 = f"""
FROM `{PROJECT}.{DATASET}.wi_county_schools`
|> WHERE agency_type = 'Public school' AND school_type = 'High School'
|> AGGREGATE COUNT(*) AS hs_count
   GROUP BY county_name, city
|> WHERE hs_count >= 3
|> AGGREGATE COUNT(*) AS city_count
   GROUP BY county_name
|> WHERE city_count >= 2
|> SELECT county_name
|> ORDER BY county_name
"""
dry_job = bq.query(sql_q8, job_config=job_config)
bytes_processed = dry_job.total_bytes_processed

tib = 1 * 1024**4
int(tib // bytes_processed)

6444892

In [27]:
#q10
sql_classify = f"""
SELECT
  school_name,
  AI.CLASSIFY(
    school_name,
    ['Person', 'Nature', 'Place', 'Other']
  ) AS predicted_category
FROM `{PROJECT}.{DATASET}.wi_county_schools`
WHERE agency_type = 'Public school'
LIMIT 50
"""

rows = list(bq.query(sql_classify).result())
{r["school_name"]: r["predicted_category"] for r in rows}

{'Webster Middle': 'Place',
 'Lincoln Elementary': 'Place',
 'Early Childhood': 'Place',
 'Amy Belle Elementary': 'Place',
 'Kettle Moraine 4K': 'Place',
 'Ozaukee Middle': 'Place',
 'Woodview Elementary': 'Place',
 'John Long Middle': 'Place',
 'Allenton Elementary': 'Place',
 'Dixon Elementary': 'Place',
 'Campbellsport High': 'Place',
 'Parkview Elementary': 'Place',
 'Rocket Academy': 'Other',
 'Cedar Grove-Belgium High': 'Place',
 'Cedar Grove-Belgium Middle': 'Place',
 'Chilton Middle': 'Place',
 'Grafton High': 'Place',
 'Rossman Elementary': 'Place',
 'Brookfield Central High': 'Place',
 'Elkhart Lake Elementary/Middle': 'Place',
 'Ozaukee High': 'Place',
 'Central Middle': 'Place',
 'Cedarburg High': 'Place',
 'Chilton High': 'Place',
 'Cleveland Elementary': 'Place',
 'MacArthur Elementary': 'Place',
 'Addison Elementary': 'Place',
 'Merton Intermediate': 'Place',
 'Swanson Elementary': 'Place',
 'Wisconsin Virtual Learning': 'Place',
 'Kennedy Elementary': 'Place',
 'Hartfor